# Agrupación en clústeres — introducción

A diferencia del aprendizaje **supervisado**, el **no supervisado** se usa cuando no hay una "verdad conocida" con la que entrenar y validar predicciones.

Su forma más común es el **clustering**, conceptualmente parecido a la clasificación salvo por una diferencia clave: los datos de entrenamiento **no incluyen el label** que se quiere predecir.

El clustering separa los casos según similitudes en sus features. Las features numéricas de una entidad pueden verse como coordenadas de un vector que define su posición en un espacio de *n* dimensiones. Lo que busca un modelo de clustering es identificar **grupos de entidades cercanas entre sí y separadas de los demás grupos**.

Trabajaremos con un dataset de medidas de distintas variedades de semilla de trigo.

> **Cita**: el dataset de semillas fue publicado originalmente por el Instituto de Agrofísica de la Academia Polaca de Ciencias en Lublin (Dua, D. y Graff, C., 2019) y está disponible en el [UCI Machine Learning Repository](http://archive.ics.uci.edu/ml).

In [ ]:
import pandas as pd

data = pd.read_csv('../datasets/seeds.csv')

# Nos quedamos solo con las features (las 6 primeras columnas)
features = data[data.columns[0:6]]
features.sample(10, random_state=0)

El dataset tiene **seis features** por cada observación de semilla. Se pueden interpretar como coordenadas que describen la posición de cada instancia en un espacio de **seis dimensiones**.

Seis dimensiones son imposibles de visualizar en un gráfico plano. Para eso usamos una técnica matemática llamada **Análisis de Componentes Principales (PCA)**: analiza las relaciones entre las features y resume cada observación en coordenadas de dos *componentes principales*. Es decir, traduce los seis valores a dos coordenadas conservando lo esencial de la estructura.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# Normalizar las features para que estén en la misma escala
scaled_features = MinMaxScaler().fit_transform(features)

# Reducir a dos componentes principales
pca = PCA(n_components=2, random_state=0).fit(scaled_features)
features_2d = pca.transform(scaled_features)

print('Forma original :', scaled_features.shape)
print('Tras PCA       :', features_2d.shape)
print(f'Varianza explicada por los 2 componentes: {pca.explained_variance_ratio_.sum():.1%}')
print()
print(features_2d[:10])

> El porcentaje de **varianza explicada** dice cuánta de la información original conservan los dos componentes. Si fuera muy bajo, la vista en 2D sería engañosa.

Ya con los datos en dos dimensiones, podemos verlos en un gráfico.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,6))
plt.scatter(features_2d[:,0], features_2d[:,1])
plt.xlabel('Dimensión 1')
plt.ylabel('Dimensión 2')
plt.title('Datos proyectados en 2D')
plt.grid(alpha=0.3)
plt.show()

Se distinguen al menos dos —y probablemente tres— grupos razonablemente separados.

Y aquí aparece **el problema fundamental del clustering**: sin labels conocidos, ¿cómo sabes en cuántos grupos dividir los datos?

## El método del codo

Una forma de averiguarlo es entrenar una serie de modelos con un número creciente de clústeres y medir **qué tan compactos** quedan los puntos dentro de cada grupo.

La métrica habitual es la **suma de cuadrados intra-clúster** (*within cluster sum of squares*, WCSS): mide la distancia de cada punto a su centroide. **Cuanto menor, más juntos están los puntos.** En scikit-learn se obtiene del atributo `inertia_`.

Al graficar el WCSS contra el número de clústeres, se busca el "codo" de la curva.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# Crear 10 modelos, de 1 a 10 clústeres
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, n_init=10, random_state=0)
    kmeans.fit(features.values)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1, 11), wcss, 'o-')
plt.title('WCSS según el número de clústeres')
plt.xlabel('Número de clústeres')
plt.ylabel('WCSS (inercia)')
plt.xticks(range(1, 11))
plt.grid(alpha=0.3)
plt.show()

for k, v in zip(range(1, 11), wcss):
    print(f'k = {k:2d}  →  WCSS = {v:,.1f}')

La curva muestra una caída grande del WCSS al pasar de uno a dos clústeres, y otra reducción notable de dos a tres. **A partir de ahí la mejora se aplana**, formando un "codo" alrededor de tres.

Esa es la señal: hay entre dos y tres grupos razonablemente bien separados.

> **Por qué funciona el método**: el WCSS *siempre* baja al añadir clústeres — con tantos clústeres como puntos, sería cero. Lo interesante no es el valor sino **dónde deja de compensar**: el punto a partir del cual añadir un grupo más ya no reduce mucho la dispersión.

## Resumen

Vimos qué significa agrupar en clústeres y cómo determinar si el clustering es apropiado para unos datos. En el cuaderno `02-kmeans-y-jerarquico.ipynb` etiquetamos los datos automáticamente con dos algoritmos distintos.